# A Verified 23.42 Portfolio for Verifiable Quantum Circuit Compilation

**Public score: 23.42** (geometric mean across the three tasks), assembled from
component circuits that each pass the competition's official verifier.

This notebook does three things:

1. Lays out the scoring model the verifiers actually implement. Most of the
   leverage in this competition comes from understanding where cost hides.
2. Hash-verifies and assembles the best verified public circuit for each task
   into a valid `submission.zip`. The output can be submitted directly.
3. Documents provenance. Every component is Apache-2.0 public work from the
   Kaggle community; the contribution here is verification discipline,
   integration, and a map of where the remaining headroom sits.

---

## 1. The scoring model

The score is a **geometric mean of three improvement ratios** (baseline cost
divided by submission cost). Because the score multiplies the three ratios,
a 10% cost cut on the weakest task is worth exactly as much as 10% on the
strongest. The weakest task is where the effort pays.

| Task | Cost model | What dominates | Public best |
|---|---|---|---|
| **invsqrt** | `Q x (Toffoli + rotation cost)` | lookup size k and Horner depth | ~20.2x |
| **dqi** | `repetition x Q^2 x T` | qubit count enters squared | ~58.9x |
| **dfthcsos** | `lambda_eff x T_total x Q_total` | lambda ~47-50, T ~17k, Q ~430 | ~9.65x |

Three consequences that drove every decision here:

* **Qubits are quadratic on dqi.** At the margin, removing a qubit beats
  removing a Toffoli roughly 2:1. The 889-qubit Schur-family decoder with
  ~434k executed Toffolis outscores narrower decoders with fewer Toffolis
  but more qubits.
* **The repetition factor punishes observed failures.** When verification
  shots fail, the cost inflates by `repetition > 1`. We measured a candidate
  dqi circuit with fewer Toffolis but 1 value error in 102,400 shots, and it
  scored worse than a zero-error circuit with more Toffolis. A slightly
  cheaper errorful circuit is a worse submission.
* **Scratch qubits must return to |0> and phases are checked.** The verifier
  tracks measurement-induced phase flips (`HMR`/`R` applied to a non-zero
  qubit flips the output sign with probability 1/2). Cleanup is part of the
  score, not an afterthought.

## 2. Component portfolio (all figures are official-verifier outputs)

| Task | Source | Qubits | Verified cost | Ratio |
|---|---|---|---|---|
| invsqrt | yoyobar 62q circuit (`bce2948ff24e`) | 62 | 100,569 | **20.65x** |
| dqi | SomeJoint-lineage confluent decoder (`fe98677c61ef`) | 888 | 3.983e11 | **61.3x** |
| dfthcsos | marculera R9/B143/C42 fit + slack-spend refit (HW-04 `02f8f7d9d75a`) | 428 | 3.310e8 | **10.14x** |

Geometric mean = (20.65 x 61.3 x 10.14)^(1/3) = **23.42**, matching the
leaderboard score returned for this exact file set (23.41982).

Each `*.verify.json` shipped in the attached dataset is the unmodified JSON
written by the competition's own verifier: shots run, error bounds, Toffoli
counts, qubit counts, and the circuit SHA-256. The assembly cell below
re-checks every file against those digests before zipping, so what gets
submitted is byte-identical to what was verified.

## 3. Assemble `submission.zip` (hash-verified)

The cell locates the three components anywhere under `/kaggle/input`,
verifies each file's SHA-256 against the digests recorded at verification
time, and writes a competition-valid `submission.zip` to `/kaggle/working`.
Any mismatch aborts the build instead of shipping a bad archive.

In [ ]:
import hashlib, json, shutil, zipfile
from pathlib import Path

# Digests recorded from the official verifier runs on this exact file set.
EXPECTED = {
    "dfthcsos/angle_k_0.kmb": "634b1f4f497e4a18c42c07acd3f4a6a96c05ed1d17a9057c346494ca6efb5b06",
    "dfthcsos/angle_k_1.kmb": "cb987c2923f823edd494462e8fc93094882dd8221af70880582e76f950bd5c69",
    "dfthcsos/angle_k_10.kmb": "060166e7ca6943721e1cab87a9b934a71eaf092cea9045c02c96f428678df632",
    "dfthcsos/angle_k_11.kmb": "85885ff299d55b8652ab5beceadbf4e4cc88f9388e72cc5001b224e5ea34ff16",
    "dfthcsos/angle_k_12.kmb": "cd3d2bb21237be6be99373e71adffbb4025e4ee93a02245aa2f9193ea612ab63",
    "dfthcsos/angle_k_13.kmb": "25fb9014a5d17dd759f955a389438acf2daecdf709f3802fa5637ca6b465ce34",
    "dfthcsos/angle_k_14.kmb": "a5c5c5be6bf87f50292a08a59cca46b393577d7829d0cf00dc175c3c548b6321",
    "dfthcsos/angle_k_2.kmb": "d5be3caac7c863fb47fedf54543e75ef680a3b9e8cf052d27e7096d035f20d7e",
    "dfthcsos/angle_k_3.kmb": "2492c3df7baa3c9973de7b65ebfe77e79fb71f1a7f20bdb3133c7b3354d2671e",
    "dfthcsos/angle_k_4.kmb": "8679e02d02a6f11d411bb5a8493cfd53271af30893c842b42d19382f6fb2257c",
    "dfthcsos/angle_k_5.kmb": "08777eccd6125ac9abcd02c6c0c7cc03c619f6f53a48d036fbe47458fb341b4d",
    "dfthcsos/angle_k_6.kmb": "603fcc2806caa7c9fb017d7feb1114653182f150a2c70f3512b6d28ea5b2ac08",
    "dfthcsos/angle_k_7.kmb": "63fbfab71c18cc185edca1d6052ae0dc7116ae4968a0c07c2114fdd63860e509",
    "dfthcsos/angle_k_8.kmb": "5a593114eb52e6659e6d61dfa21a305e27288346e30fd2e8df96bddcded3f381",
    "dfthcsos/angle_k_9.kmb": "811f8864ba26f3059408b4f41af9e26877861eada9747d2a80523ddd450e4432",
    "dqi.kmb": "fe98677c61efd6f91ca7967254d8d415a23080326c1659c146bdb30c1915dedb",
    "invsqrt.kmb": "bce2948ff24e20fe59666d131f7fc4316a905103937272c52fdd39d494b6fe47",
    "dfthcsos/submission.npz": "8ea801a126127113865e3f9488ad35bcdc9091265b6e23d8b15a637a827b2c74"
}

def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

WORK = Path("/kaggle/working")
SUB  = WORK / "submission"
if SUB.exists():
    shutil.rmtree(SUB)

by_name = {}
for p in Path("/kaggle/input").rglob("*"):
    if p.is_file():
        by_name.setdefault(p.name, p)

missing, bad = [], []
print("verifying component digests:")
for rel, want in EXPECTED.items():
    name = rel.split("/")[-1]
    src = by_name.get(name)
    if src is None:
        missing.append(rel); continue
    got = sha256(src)
    status = "ok  " if got == want else "FAIL"
    print(f"  {status} {rel:28s} {got[:16]}...")
    if got != want:
        bad.append(rel); continue
    dest = SUB / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dest)

if missing:
    print("\nMISSING (attach the portfolio dataset):", missing)
if bad:
    print("\nHASH MISMATCHES - refusing to build:", bad)
if missing or bad:
    raise SystemExit("assembly aborted: inputs did not match verified digests")

zpath = WORK / "submission.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(SUB.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(SUB))
names = zipfile.ZipFile(zpath).namelist()
print(f"\nwrote {zpath} ({zpath.stat().st_size/1e6:.1f} MB)")
assert "invsqrt.kmb" in names and "dqi.kmb" in names and "dfthcsos/submission.npz" in names
print("layout OK - ready to submit")

## 4. Verification records

The unmodified verifier outputs bundled with the dataset. Note the dqi
entry: `value_errors: 0`, `phase_errors: 0` across 102,400 shots. That
zero-error property is what the repetition-factor math rewards.

In [ ]:
import json
from pathlib import Path

for jf in Path("/kaggle/input").rglob("*.verify.json"):
    rec = json.loads(jf.read_text())
    keep = {k: rec.get(k) for k in (
        "status","score","num_qubits","toffoli_count","verified_shots",
        "value_errors","phase_errors","repetition_factor",
        "lambda_eff","error_1","error_2","circuit_sha256") if k in rec}
    print(jf.name); print(json.dumps(keep, indent=2)[:1400]); print()

# implied geometric mean from the verified costs
BASE = {"invsqrt": 2_076_774.0, "dqi": 2.4425e13, "dfthcsos": 3.355599588e9}
try:
    scores = {}
    for jf in Path("/kaggle/input").rglob("*.verify.json"):
        rec = json.loads(jf.read_text())
        for task in BASE:
            if jf.name.startswith(task) and rec.get("status") == "PASS":
                scores[task] = BASE[task] / rec["score"]
    if len(scores) == 3:
        gm = (scores["invsqrt"] * scores["dqi"] * scores["dfthcsos"]) ** (1/3)
        print(f"implied GM from verifier records: {gm:.2f}x "
              f"(invsqrt {scores['invsqrt']:.2f} x dqi {scores['dqi']:.2f} x chem {scores['dfthcsos']:.2f})")
except Exception as e:
    print("gm calc skipped:", e)

## 6. Provenance and license

Components are assembled from Apache-2.0 public releases by the community.
Credit goes to yoyobar for the 62-qubit invsqrt circuit, the SomeJoint
decoder lineage for the 888-qubit DQI circuit, and marculera for the
R9/B143/C42 FeMoCo factorization, with the slack-spending weight refit
(Hedge Wizardry HW-04) by manwithacat. Per-file digests in the verifier
records identify exactly which artifacts are used. The value added here
is verification discipline, integration, and the scoring-model map in
section 1.